<div style="background: linear-gradient(135deg, #1d4ed8, #3b82f6); padding: 2rem; border-radius: 16px; color: white; text-align: center;">
  <h1 style="font-size: 2.5rem; margin: 0;">&#x1F697; Drive Wise</h1>
  <p style="font-size: 1.2rem; margin: 0.5rem 0 0 0; opacity: 0.9;">Metadata-Aware Automotive RAG Assistant</p>
  <p style="font-size: 0.9rem; margin: 0.5rem 0 0 0; opacity: 0.75;">Powered by Google Gemini 2.5 Flash &middot; Hybrid Semantic + Keyword Retrieval</p>
</div>

---

## What This Notebook Does

1. **3 sample car brochures are included** - ready to use immediately (Mahindra XUV700, Hyundai Aura, Hyundai Grand I10 Nios)
2. **You can upload your own PDF brochures** - name them `Brand_Model.pdf` (e.g. `Toyota_Fortuner.pdf`)
3. **Ask any question** about any indexed car - get grounded answers with citations

> **Quick Start**: Click **Runtime -> Run All**, add your `GOOGLE_API_KEY` in the Secrets panel (left sidebar), then scroll to the bottom to chat!

## Step 1 — Install Dependencies

In [ ]:
!pip install -q google-generativeai pypdf numpy ipywidgets
print("All packages installed!")

## Step 2 — Configure Your Gemini API Key

Get a **free** key at [aistudio.google.com/apikey](https://aistudio.google.com/apikey)

**On Colab**: Click the **Secrets** icon (left sidebar, looks like a key) → add `GOOGLE_API_KEY`

In [ ]:
import os, json, time, hashlib
import numpy as np
import google.generativeai as genai
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
from pypdf import PdfReader

api_key = None
try:
    from google.colab import userdata
    api_key = userdata.get('GOOGLE_API_KEY')
    print("API key loaded from Colab Secrets.")
except Exception:
    pass

if not api_key:
    api_key = os.environ.get('GOOGLE_API_KEY')
    if api_key:
        print("API key loaded from environment variable.")

if not api_key:
    api_key = "YOUR_GEMINI_API_KEY_HERE"  # Paste here if needed
    print("WARNING: No API key found. Set GOOGLE_API_KEY in Colab Secrets.")

genai.configure(api_key=api_key)
print("Gemini API configured!")

## Step 3 — Download Sample Brochures & Build Index

Downloads 3 sample PDFs directly from GitHub and indexes them (~2 minutes).

In [ ]:
import urllib.request

SECTION_KEYWORDS = {
    "Engine & Performance": ["engine","torque","power","gearbox","transmission","hp","ps","cc",
        "cylinder","performance","speed","acceleration","manual","automatic","dct","cvt","bhp","rpm"],
    "Mileage & Fuel Efficiency": ["mileage","fuel economy","fuel efficiency","kmpl","km/l",
        "consumption","hybrid","electric range","efficiency","co2","emissions","arai","wltp"],
    "Safety": ["safety","airbag","abs","ebd","esc","brake","crash test","ncap","adas",
        "lane assist","isofix","hill assist","esp","traction control","rear view camera","tpms"],
    "Dimensions": ["dimensions","length","width","height","wheelbase","ground clearance",
        "boot space","weight","capacity","turning radius","fuel tank","kerb weight","mm"],
    "Interior & Comfort": ["interior","comfort","seat","upholstery","climate control","ac",
        "sunroof","steering","cabin","leather","ventilated","ambient lighting","armrest","cruise control"],
    "Infotainment & Connectivity": ["infotainment","screen","display","apple carplay","android auto",
        "bluetooth","speakers","audio","navigation","connected car","usb","voice command","touchscreen"],
}

def classify_section(text):
    scores = {sec: 0 for sec in SECTION_KEYWORDS}
    tl = text.lower()
    for sec, kws in SECTION_KEYWORDS.items():
        for kw in kws:
            scores[sec] += tl.count(kw)
    best = max(scores, key=scores.get)
    return best if scores[best] >= 2 else "General Specifications"

def extract_brand_model(filename):
    name = os.path.splitext(os.path.basename(filename))[0]
    for sep in ["_", "-"]:
        if sep in name:
            parts = name.split(sep)
            return parts[0].strip().title(), " ".join(parts[1:]).strip().title()
    return "Unknown", name.title()

def chunk_pdf(filepath):
    reader = PdfReader(filepath)
    brand, model = extract_brand_model(filepath)
    chunks = []
    for page_idx, page in enumerate(reader.pages):
        text = page.extract_text()
        if not text or not text.strip():
            continue
        paras = text.split("\n\n")
        current, current_len = [], 0
        for para in paras:
            para = para.strip()
            if not para:
                continue
            if current_len + len(para) > 800 and current:
                chunk_text = "\n".join(current)
                chunks.append({"text": chunk_text, "brand": brand, "model": model,
                    "section": classify_section(chunk_text),
                    "page": page_idx + 1, "source_file": os.path.basename(filepath)})
                current, current_len = [para], len(para)
            else:
                current.append(para)
                current_len += len(para)
        if current:
            chunk_text = "\n".join(current)
            chunks.append({"text": chunk_text, "brand": brand, "model": model,
                "section": classify_section(chunk_text),
                "page": page_idx + 1, "source_file": os.path.basename(filepath)})
    return chunks, brand, model

def embed_chunks(chunks):
    """Embeds chunks in batches with full retry on all transient errors."""
    BATCH = 10  # smaller batch to reduce chance of timeouts
    TRANSIENT = ("429", "500", "503", "connection", "timeout", "remote", "aborted", "reset")
    for i in range(0, len(chunks), BATCH):
        batch = chunks[i:i+BATCH]
        contents = [
            f"Car Brand: {c['brand']}\nModel: {c['model']}\nSection: {c['section']}\nPage: {c['page']}\n{c['text']}"
            for c in batch
        ]
        last_err = None
        for attempt in range(6):
            try:
                resp = genai.embed_content(model='models/gemini-embedding-001', content=contents)
                for j, emb in enumerate(resp['embedding']):
                    batch[j]['embedding'] = np.array(emb, dtype=np.float32)
                last_err = None
                break
            except Exception as e:
                last_err = e
                err_lower = str(e).lower()
                is_transient = any(t in err_lower for t in TRANSIENT)
                if is_transient and attempt < 5:
                    wait = 5 * (2 ** attempt)  # 5, 10, 20, 40, 80 seconds
                    print(f"  Network/rate error (attempt {attempt+1}/6) - retrying in {wait}s: {str(e)[:60]}")
                    time.sleep(wait)
                else:
                    raise
        if last_err:
            raise last_err
        time.sleep(1.0)  # small pause between batches

index_data = {"files": {}, "chunks": []}

def index_pdf(filepath):
    fname = os.path.basename(filepath)
    fhash = hashlib.md5(open(filepath,'rb').read()).hexdigest()
    if fname in index_data["files"] and index_data["files"][fname].get("hash") == fhash:
        print(f"  {fname} already indexed - skipping.")
        return
    index_data["chunks"] = [c for c in index_data["chunks"] if c.get("source_file") != fname]
    print(f"  Chunking {fname}...")
    chunks, brand, model = chunk_pdf(filepath)
    print(f"     Got {len(chunks)} chunks. Generating embeddings...")
    embed_chunks(chunks)
    index_data["chunks"].extend(chunks)
    index_data["files"][fname] = {"hash": fhash, "brand": brand, "model": model, "chunks_count": len(chunks)}
    print(f"  Done: {brand} {model} ({len(chunks)} chunks)")

SAMPLES_BASE = "https://raw.githubusercontent.com/avanishar/drivewiseapp/main/samples"
SAMPLE_FILES = ["Mahindra_XUV700.pdf", "Hyundai_Aura.pdf", "Hyundai_GrandI10Nios.pdf"]

os.makedirs("brochures", exist_ok=True)
print("Downloading sample brochures...")
for fname in SAMPLE_FILES:
    fpath = f"brochures/{fname}"
    if not os.path.exists(fpath):
        url = f"{SAMPLES_BASE}/{fname}"
        urllib.request.urlretrieve(url, fpath)
        size_kb = os.path.getsize(fpath) / 1024
        if size_kb < 10:
            os.remove(fpath)
            print(f"  FAILED: {fname} too small ({size_kb:.1f} KB) - may be LFS pointer")
        else:
            print(f"  Downloaded: {fname} ({size_kb/1024:.2f} MB)")
    else:
        print(f"  Already present: {fname}")

print("\nBuilding index from sample brochures (takes ~2 min)...")
for fname in os.listdir("brochures"):
    if fname.lower().endswith(".pdf"):
        index_pdf(f"brochures/{fname}")

print(f"\nIndex ready! {len(index_data['chunks'])} chunks from {len(index_data['files'])} car(s).")

## Step 4 — (Optional) Upload Your Own Brochure PDF

Upload any car brochure in PDF format. Name it `Brand_Model.pdf` (e.g. `Toyota_Fortuner.pdf`). It will be indexed and appear in the chat dropdown.

In [ ]:
try:
    from google.colab import files as colab_files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

upload_status = widgets.HTML(value='')

if IN_COLAB:
    upload_btn = widgets.Button(
        description='Upload PDF Brochure',
        button_style='info', icon='upload',
        layout=widgets.Layout(width='220px', height='40px')
    )
    def on_upload(b):
        upload_status.value = '<span style="color:#3b82f6;">Opening file picker...</span>'
        uploaded = colab_files.upload()
        if not uploaded:
            upload_status.value = '<span style="color:#64748b;">No file selected.</span>'
            return
        for filename, content in uploaded.items():
            if not filename.lower().endswith('.pdf'):
                upload_status.value = f'<span style="color:#ef4444;">Not a PDF: {filename}</span>'
                continue
            fpath = f"brochures/{filename}"
            with open(fpath, 'wb') as f:
                f.write(content)
            upload_status.value = f'<span style="color:#3b82f6;">Indexing {filename}... (~1-2 min)</span>'
            try:
                index_pdf(fpath)
                new_brands = sorted(set(m.get('brand') for m in index_data['files'].values()))
                brand_dd.options = new_brands
                upload_status.value = f'<span style="color:#10b981;">Done! {filename} indexed. Select it in chat below.</span>'
            except Exception as e:
                upload_status.value = f'<span style="color:#ef4444;">Error: {e}</span>'
    upload_btn.on_click(on_upload)
    display(widgets.VBox([
        widgets.HTML(
            '<div style="background:#eff6ff;border:1px solid #bfdbfe;border-radius:10px;padding:12px 16px;">'
            '<b>Upload Your Own Car Brochure PDF</b><br>'
            '<span style="font-size:0.85rem;color:#475569;">Name it Brand_Model.pdf '
            '(e.g. Toyota_Fortuner.pdf). It will be indexed and available in the chat.</span></div>'
        ),
        upload_btn,
        upload_status
    ]))
else:
    print("Running locally: place your PDF in the brochures/ folder and re-run the indexing cell.")

## Step 5 — RAG Engine & Generator

In [ ]:
STOPWORDS = {
    "a","about","above","after","again","all","am","an","and","any","are","as","at",
    "be","because","been","before","being","below","between","both","but","by","can",
    "did","do","does","doing","down","during","each","for","from","had","has","have",
    "having","he","her","here","him","his","how","i","if","in","into","is","it","its",
    "me","more","most","my","no","nor","not","of","off","on","once","only","or","other",
    "our","out","over","own","same","she","should","so","some","such","than","that",
    "the","their","them","then","there","these","they","this","those","through","to",
    "too","under","until","up","very","was","we","were","what","when","where","which",
    "while","who","whom","why","with","you","your"
}

def cosine_sim(v1, v2):
    n1, n2 = np.linalg.norm(v1), np.linalg.norm(v2)
    return float(np.dot(v1, v2) / (n1 * n2)) if n1 > 0 and n2 > 0 else 0.0

def kw_score(query, text):
    words = [w.strip("?,.:;!\"'()").lower() for w in query.split() if w.lower() not in STOPWORDS]
    if not words: return 0.0
    tl = text.lower()
    return sum(1 for w in words if w in tl) / len(words)

def retrieve(query, brand, model, limit=4):
    filtered = [c for c in index_data['chunks']
                if c.get('brand','').lower() == brand.lower()
                and c.get('model','').lower() == model.lower()]
    if not filtered:
        return []
    emb_resp = genai.embed_content(model='models/gemini-embedding-001', content=query)
    q_emb = emb_resp['embedding']
    scored = []
    for c in filtered:
        emb = c.get('embedding')
        if emb is None: continue
        sem = cosine_sim(q_emb, emb)
        kw  = kw_score(query, c['text'])
        scored.append({**c, 'score': 0.8*sem + 0.2*kw})
    scored.sort(key=lambda x: x['score'], reverse=True)
    return scored[:limit]

def generate_answer(query, brand, model):
    chunks = retrieve(query, brand, model)
    if not chunks:
        return f"No brochure data found for '{brand} {model}'.", []
    context_str = "".join(
        f"\n--- Source [{i+1}] (Page {c['page']}, Section: {c['section']}) ---\n{c['text']}\n"
        for i, c in enumerate(chunks)
    )
    sys_prompt = (
        f"You are an expert automotive assistant for Drive Wise. "
        f"Answer ONLY from the brochure excerpts for {brand} {model}. "
        "Rules: 1) Use only the provided context. 2) If missing, say so. "
        "3) Use inline citations [1],[2] matching source numbers. 4) Be clear and concise."
    )
    prompt = f"Brochure Context for {brand} {model}:\n{context_str}\nUser Query: \"{query}\"\nGrounded Answer:"
    try:
        resp = genai.GenerativeModel(
            model_name="models/gemini-2.5-flash",
            system_instruction=sys_prompt
        ).generate_content(prompt, generation_config=genai.types.GenerationConfig(temperature=0.1))
        return resp.text.strip(), chunks
    except Exception as e:
        return f"Error: {e}", []

def get_car_map():
    cm = {}
    for meta in index_data['files'].values():
        b, m = meta.get('brand','Unknown'), meta.get('model','Unknown')
        if b not in cm: cm[b] = []
        if m not in cm[b]: cm[b].append(m)
    return cm

print("RAG engine ready!")

---
# Interactive Chat — Ask Your Own Questions!

- Select **Brand** and **Model** from the dropdowns
- Type your question
- Click **Ask DriveWise**

> Uploaded a new brochure? It will appear in the Brand dropdown automatically!

In [ ]:
chat_history = []
CAR_MAP = get_car_map()
brand_list = sorted(CAR_MAP.keys())

brand_dd = widgets.Dropdown(
    options=brand_list,
    value=brand_list[0] if brand_list else None,
    description='Brand:',
    style={'description_width': '55px'},
    layout=widgets.Layout(width='220px')
)
model_dd = widgets.Dropdown(
    options=CAR_MAP.get(brand_list[0], []) if brand_list else [],
    description='Model:',
    style={'description_width': '55px'},
    layout=widgets.Layout(width='220px')
)
query_box = widgets.Textarea(
    placeholder='Type your question here...\nExamples:\n- What safety features does it have?\n- What is the mileage?\n- What are the engine specs?\n- What is the boot space and seating?',
    layout=widgets.Layout(width='100%', height='90px')
)
ask_btn = widgets.Button(
    description='Ask DriveWise',
    button_style='primary', icon='search',
    layout=widgets.Layout(width='165px', height='38px')
)
clear_btn = widgets.Button(
    description='Clear Chat',
    button_style='warning', icon='trash',
    layout=widgets.Layout(width='130px', height='38px')
)
status_lbl = widgets.HTML(
    value='<span style="color:#64748b;font-size:0.88rem;">Ready! Select a car and ask a question.</span>'
)
chat_out = widgets.Output(layout=widgets.Layout(
    border='1px solid #e2e8f0', border_radius='10px', padding='14px',
    min_height='180px', max_height='520px', overflow_y='auto'
))

def on_brand_change(change):
    model_dd.options = get_car_map().get(change['new'], [])

brand_dd.observe(on_brand_change, names='value')

def render_chat():
    with chat_out:
        clear_output(wait=True)
        if not chat_history:
            display(HTML(
                '<div style="text-align:center;padding:36px;color:#94a3b8;">'
                '<div style="font-size:3rem;">&#x1F697;</div>'
                '<div style="font-size:1.05rem;margin-top:8px;font-weight:600;">Ask anything about your car!</div>'
                '<div style="font-size:0.82rem;margin-top:6px;color:#cbd5e1;">'
                'mileage &bull; safety features &bull; engine specs &bull; dimensions &bull; infotainment'
                '</div></div>'
            ))
            return
        for item in chat_history:
            display(HTML(
                '<div style="margin:10px 0;display:flex;justify-content:flex-end;">'
                '<div style="background:#eff6ff;border:1px solid #bfdbfe;border-radius:16px 16px 4px 16px;'
                'padding:10px 14px;max-width:80%;font-size:0.92rem;color:#1e3a5f;">'
                f'<b>You</b> <span style="font-size:0.75rem;color:#94a3b8;">({item["car"]})</span><br>'
                f'{item["query"]}</div></div>'
            ))
            answer_html = item['answer'].replace('\n', '<br>')
            display(HTML(
                '<div style="margin:6px 0 12px 0;">'
                '<div style="background:#fff;border:1px solid #e2e8f0;border-left:4px solid #3b82f6;'
                'border-radius:4px 14px 14px 14px;padding:12px 16px;max-width:95%;'
                'font-size:0.92rem;color:#1e293b;box-shadow:0 2px 6px rgba(0,0,0,0.05);">'
                f'<b>Drive Wise</b> <span style="font-size:0.75rem;color:#64748b;">({item["time"]}s)</span><br><br>'
                f'{answer_html}</div></div>'
            ))
            if item.get('sources'):
                tags = "".join(
                    f'<span style="display:inline-block;background:#f0fdf4;color:#166534;'
                    f'font-size:0.73rem;padding:2px 8px;border-radius:20px;margin:2px;'
                    f'border:1px solid #bbf7d0;">[{i+1}] Pg {s["page"]} - {s["section"]}</span>'
                    for i, s in enumerate(item['sources'])
                )
                display(HTML(f'<div style="margin:-4px 0 10px 12px;font-size:0.8rem;">Sources: {tags}</div>'))
            display(HTML("<hr style='border:none;border-top:1px solid #f1f5f9;margin:4px 0;'>"))

def on_ask(b):
    query = query_box.value.strip()
    if not query:
        status_lbl.value = '<span style="color:#ef4444;">Please type a question first!</span>'
        return
    if not brand_dd.value or not model_dd.value:
        status_lbl.value = '<span style="color:#ef4444;">No car indexed yet. Run Step 3 first!</span>'
        return
    brand, model = brand_dd.value, model_dd.value
    ask_btn.disabled = clear_btn.disabled = True
    ask_btn.description = 'Thinking...'
    status_lbl.value = f'<span style="color:#3b82f6;">Searching brochure for {brand} {model}...</span>'
    query_box.value = ''
    t0 = time.time()
    answer, sources = generate_answer(query, brand, model)
    elapsed = round(time.time() - t0, 2)
    chat_history.append({'query': query, 'answer': answer, 'sources': sources,
                         'car': f'{brand} {model}', 'time': elapsed})
    render_chat()
    status_lbl.value = f'<span style="color:#10b981;">Done in {elapsed}s - used {len(sources)} source chunk(s)</span>'
    ask_btn.disabled = clear_btn.disabled = False
    ask_btn.description = 'Ask DriveWise'

def on_clear(b):
    chat_history.clear()
    render_chat()
    status_lbl.value = '<span style="color:#64748b;">Chat cleared.</span>'

ask_btn.on_click(on_ask)
clear_btn.on_click(on_clear)

render_chat()
display(widgets.VBox([
    widgets.HTML(
        '<div style="background:linear-gradient(135deg,#1d4ed8,#3b82f6);color:white;'
        'padding:14px 20px;border-radius:12px 12px 0 0;">'
        '<span style="font-size:1.2rem;font-weight:700;">Drive Wise - Interactive Chat</span>'
        '<span style="font-size:0.82rem;opacity:0.8;margin-left:10px;">Grounded answers from car brochures</span>'
        '</div>'
    ),
    widgets.HBox([brand_dd, model_dd], layout=widgets.Layout(gap='10px', margin='8px 0')),
    widgets.HTML('<div style="font-weight:600;color:#334155;margin:4px 0 2px 0;">Your Question:</div>'),
    query_box,
    widgets.HBox([ask_btn, clear_btn, status_lbl],
                 layout=widgets.Layout(gap='10px', align_items='center', margin='6px 0')),
    chat_out
], layout=widgets.Layout(
    border='1px solid #e2e8f0', border_radius='12px',
    padding='16px', width='100%'
)))

---
<div style="background:#f0fdf4;border:1px solid #86efac;border-radius:12px;padding:1.2rem;text-align:center;">
<h3 style="color:#166534;margin:0;">DriveWise RAG Demo Complete!</h3>
<p style="color:#15803d;margin:0.4rem 0 0 0;font-size:0.95rem;">
Full pipeline: PDF chunking &rarr; Gemini embeddings &rarr; hybrid retrieval &rarr; grounded generation
</p>
</div>

### Key Design Decisions

| Component | Choice | Reason |
|-----------|--------|---------|
| Embeddings | `gemini-embedding-001` | High-quality multilingual embeddings |
| Generation | `gemini-2.5-flash` | Fast, cost-efficient, strong instruction following |
| Retrieval | Hybrid (80% semantic + 20% keyword) | Captures meaning AND exact spec numbers |
| Chunking | Paragraph-level (~800 chars) | Balances context and precision |
| Section Tags | Rule-based keyword scoring | Metadata pre-filter reduces noise |